In [ ]:
import os
import re
import sys

import qubx
from qubx import logger

%qubxd

%load_ext autoreload
%autoreload 2

from qubx.backtester.simulated_data import EmulatedBarSequence
from qubx.backtester.simulator import simulate
from qubx.core.basics import DataType, Signal, TimestampedDict
from qubx.core.interfaces import IStrategy, IStrategyContext, IStrategyInitializer, MarketEvent, TriggerEvent
from qubx.core.lookups import lookup
from qubx.data.registry import StorageRegistry
from qubx.data.transformers import TypedGenericSeries
from qubx.utils.time import handle_start_stop


⠀⠀⡰⡖⠒⠒⢒⢦⠀⠀   
⠀⢠⠃⠈⢆⣀⣎⣀⣱⡀  QUBX | Quantitative Backtesting Environment 
⠀⢳⠒⠒⡞⠚⡄⠀⡰⠁         (c) 2025, ver. 0.7.40.dev8
⠀⠀⠱⣜⣀⣀⣈⣦⠃⠀⠀⠀ 
        


In [2]:
stor = StorageRegistry.get("csv::../../tests/data/storages/csv/")
stor.get_exchanges()

['BINANCE.UM', 'HYPERLIQUID', 'KRAKEN']

In [3]:
stor.get_reader("BINANCE.UM", "SWAP").get_time_range("AAVEUSDT", "ohlc(1h)")

(numpy.datetime64('2023-06-01T00:00:00'),
 numpy.datetime64('2023-08-01T00:00:00'))

In [4]:
stor.get_reader("BINANCE.UM", "SWAP").get_time_range("BTCUSDT", "quote")

(numpy.datetime64('2017-08-24T13:01:12'),
 numpy.datetime64('2017-08-24T13:01:59'))

In [5]:
stor.get_reader("KRAKEN", "SWAP").get_time_range("AAVEUSD", "ohlc(1h)")

(numpy.datetime64('2023-06-01T00:00:00'),
 numpy.datetime64('2023-08-01T00:00:00'))

In [6]:
stor.get_reader("KRAKEN", "SWAP").read("BTCUSD", "quote").to_records()[:3]

[[2025-12-01T00:00:00.000000000]	90388.00000 (0.01) | 90389.00000 (0.01),
 [2025-12-01T00:00:01.000000000]	90380.00000 (0.01) | 90386.00000 (0.01),
 [2025-12-01T00:00:02.000000000]	90364.00000 (0.01) | 90370.00000 (0.01)]

In [ ]:
stor.get_reader("KRAKEN", "SWAP").read("BTCUSD", "quote", None, None).transform(TypedGenericSeries())

In [11]:
_X1 = pd.DataFrame.from_dict({
    "2025-01-01 00:00": {"open": 1000, "high": 1100, "low": 950, "close": 1050 },
    "2025-01-01 01:00": {"open": 1050, "high": 1200, "low": 900, "close": 1075 },
    "2025-01-01 02:00": {"open": 1075, "high": 1300, "low": 1050, "close": 1100 },
}, orient="index")
_X1.index = pd.DatetimeIndex(_X1.index)

_X2 = pd.DataFrame.from_dict({
    "2025-01-01 00:00": {"open": 100, "high": 110, "low": 95, "close": 105 },
    "2025-01-01 01:00": {"open": 105, "high": 120, "low": 90, "close": 107 },
    "2025-01-01 02:00": {"open": 107, "high": 130, "low": 105, "close": 110 },
}, orient="index")
_X2.index = pd.DatetimeIndex(_X2.index)

In [12]:
stor = StorageRegistry.get("handy", data={
    "BINANCE.UM:SWAP:BTCUSDT": _X1,
    "BINANCE.UM:SWAP:ETHUSDT": _X2
})
r = stor["BINANCE.UM", "SWAP"]

In [13]:
r.read(["BTCUSDT", "ETHUSDT"], "ohlc(1h)")

-[MULTI DATA]-
 | BTCUSDT(ohlc(1h))[2025-01-01 00:00:00 : 2025-01-01 02:00:00] : (3 x 5)
 | ETHUSDT(ohlc(1h))[2025-01-01 00:00:00 : 2025-01-01 02:00:00] : (3 x 5)

In [56]:
from qubx.data.containers import IteratorsMaster, IteratorsMasterWithTransformer, IteratorWithTransformer

In [74]:
xr = stor.get_reader("BINANCE.UM", "SWAP")

In [108]:
d1 = xr.read(["BTCUSDT"], "ohlc(1h)", chunksize=1000)
d2 = xr.read(["ETHUSDT"], "ohlc(1h)", chunksize=1000)

In [93]:
xr.get_time_range("BTCUSDT", "ohlc(1h)")

(numpy.datetime64('2023-06-01T00:00:00'),
 numpy.datetime64('2023-08-01T00:00:00'))

In [91]:
next(d1).to_pd().tail()

StopIteration: 

In [114]:
d1 = xr.read(["BTCUSDT"], "ohlc(1h)", chunksize=1000)
ix = IteratorWithTransformer(d1, EmulatedBarSequence())

In [112]:
next(ix)[-10:]

KeyError: slice(-10, None, None)

In [117]:
next(ix)["BTCUSDT"][-10:]

StopIteration: 

In [47]:
dd = next(IteratorsMaster(d1.iterators + d2.iterators))

In [71]:
next(IteratorsMasterWithTransformer(d1.iterators + d2.iterators, EmulatedBarSequence()))

{'BTCUSDT': [[2023-06-06T00:00:01.000000000] {o:25714.600000 | h:25714.600000 | l:25714.600000 | c:25714.600000 | v:0.000000},
  [2023-06-06T00:24:00.000000000] {o:25714.600000 | h:25741.700000 | l:25714.600000 | c:25741.700000 | v:0.000000},
  [2023-06-06T00:36:00.000000000] {o:25714.600000 | h:25741.700000 | l:25660.000000 | c:25660.000000 | v:0.000000},
  [2023-06-06T00:59:59.000000000] {o:25714.600000 | h:25741.700000 | l:25660.000000 | c:25679.200000 | v:14823.830000},
  [2023-06-06T01:00:01.000000000] {o:25679.200000 | h:25679.200000 | l:25679.200000 | c:25679.200000 | v:0.000000},
  [2023-06-06T01:24:00.000000000] {o:25679.200000 | h:25687.400000 | l:25679.200000 | c:25687.400000 | v:0.000000},
  [2023-06-06T01:36:00.000000000] {o:25679.200000 | h:25687.400000 | l:25601.000000 | c:25601.000000 | v:0.000000},
  [2023-06-06T01:59:59.000000000] {o:25679.200000 | h:25687.400000 | l:25601.000000 | c:25629.600000 | v:10989.461000},
  [2023-06-06T02:00:01.000000000] {o:25629.600000 | h

In [14]:
class Test1(IStrategy):
    def on_init(self, initializer: IStrategyInitializer):
        pass

    def on_market_data(self, ctx: IStrategyContext, data: MarketEvent) -> list[Signal] | Signal | None:
        pass

In [30]:
DataType.from_str("ohlc_quotes(1h)")

(ohlc_quotes, {'timeframe': '1h'})

In [36]:
handle_start_stop("-1d", None)

ValueError: First argument is delta but stop time is not defined !

In [ ]:
simulate(
    Test1(), 
    { "ohlc": stor, "quote": stor, "some": stor}, 
    # stor,
    capital=1000, start="2020-01-01",
    instruments=[
        "BINANCE.UM:SWAP:BTCUSDT",
        "HYPERLIQUID:SWAP:BTCUSDC"
    ], 
    debug="DEBUG"
)

In [33]:
xs = StorageRegistry.get("qdb::quantlab")

In [34]:
rs = xs.get_reader("HYPERLIQUID", "SWAP")
rb = xs.get_reader("BINANCE.UM", "SWAP")

In [36]:
rs.get_data_types("BTCUSDC")

[funding_payment,
 orderbook,
 'ohlc(1min)',
 'ohlc(2min)',
 'ohlc(3min)',
 'ohlc(5min)',
 'ohlc(10min)',
 'ohlc(15min)',
 'ohlc(30min)',
 'ohlc(1h)',
 'ohlc(2h)',
 'ohlc(3h)',
 'ohlc(4h)',
 'ohlc(6h)',
 'ohlc(8h)',
 'ohlc(12h)',
 'ohlc(1d)',
 'ohlc(1w)',
 open_interest,
 quote,
 funding_rate]

In [48]:
for x in rs.read(["BTCUSDC", "AAVEUSDC", "ETHUSDC", "LTCUSDC"], "ohlc(1h)", "2025-01-01", "2026-01-01 00:10"):
    # x.to_pd().to_csv(f"../../tests/data/storages/csv/HYPERLIQUID/SWAP/{x.data_id}.1H.csv.gz")
    print(x)

LTCUSDC(ohlc(1h))[2025-01-01 00:00:00 : 2026-01-01 00:00:00] : (8761 x 11)
ETHUSDC(ohlc(1h))[2025-01-01 00:00:00 : 2026-01-01 00:00:00] : (8761 x 11)
BTCUSDC(ohlc(1h))[2025-01-01 00:00:00 : 2026-01-01 00:00:00] : (8761 x 11)
AAVEUSDC(ohlc(1h))[2025-01-01 00:00:00 : 2026-01-01 00:00:00] : (8761 x 11)


In [53]:
xx1 = rs.read(["BTCUSDC", "AAVEUSDC", "ETHUSDC", "LTCUSDC"], "ohlc(1h)", "2025-01-01", "2026-01-01 00:10")

In [ ]:
xx1.to_pd(1).head()